# Object detection with YOLO (You Only Look Once)

⚠️ IMPORTANT: Set the Runtime to GPU

To ensure this code runs correctly, you must enable the GPU:

1. Click the triangle icon (Runtime) in the right side of the top panel.

2. Select `Change runtime type`.

3. Under `Hardware accelerator`, select `T4 GPU` (or your preferred GPU).

4. Click Save.

Please take a look at the code we have prepared. This notebook demonstrates how to train a YOLO model on a prepared dataset. Because most of the code are simple operations to download and adjust the dataset for training and visualization, while the training process itself is condensed into a single method call, we have pre-written the implementation for you.

However, we have included comments throughout every step. Even though you are executing the cells rather than writing the code yourself, take your time examine these comments and understand the logic behind each step.

# Download the dataset and unzip it

The dataset is available at: <add_link_here>

It consists of synthetic images created for object detection tasks, featuring five target classes: a cactus, a traffic cone, a fire hydrant, a pallet, and a traffic light. The images were generated using NVIDIA Isaac Sim, where the target objects were spawned alongside various assets from the YCB Object and Model Set (https://www.ycbbenchmarks.com/) acting as environmental obstacles. The dataset includes automatically generated bounding box annotations in YOLO format, making it ready for immediate use in training object detection models.

In [ ]:
# Download the dataset
!dataset_url='<add_link_here>'

!wget -O dataset.zip $dataset_url

In [ ]:
# Unzip the dataset
import zipfile

with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('extracted_files')

# Visualize the structure of the dataset

In [ ]:
# Visualize the structure of the dataset
!apt-get install tree -y > /dev/null
!tree -d extracted_files

# Update the paths in data.yaml file (required in Google Colab)

In [ ]:
# Update the data yaml file to have proper format
import yaml

yaml_path = '/content/extracted_files/FOSSBot Object Detection/yolo_dataset/data.yaml'

# Read the existing yaml
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Add a 'path' key pointing to the root folder of the dataset
data['path'] = '/content/extracted_files/FOSSBot Object Detection/yolo_dataset'

# Overwrite it back
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("data.yaml updated for Ultralytics YOLO!")

# Install ultralytics library

In [ ]:
# Install the ultralytics package
!pip install ultralytics

In [ ]:
!rm -f "/content/extracted_files/FOSSBot Object Detection dataset/yolo_dataset/labels/train.cache"
!rm -f "/content/extracted_files/FOSSBot Object Detection dataset/yolo_dataset/labels/val.cache"
!rm -f "/content/extracted_files/FOSSBot Object Detection dataset/yolo_dataset/labels/test.cache"

# Training

The cell below runs the training of the YOLO model. As you can see, it involves creating a YOLO class object and running the train method with the appropriate arguments:
- `data`: the path to the `data.yaml` file;
- `epochs`: the number of training epochs. We set this to 25, as the results were sufficient while keeping the training time under 10 minutes;
- `imgsz`: the image size;
- `batch`: the batch size (the number of images processed simultaneously);
- `device`: important when running on a workstation with multiple GPUs, allowing you to select a specific one;
- `workers`: the number of CPU cores to use;
- `cache`: a flag indicating whether to cache images and labels (caching loads images into memory for faster access, reducing overall training time).

In [ ]:
from ultralytics import YOLO

# Load a pretrained lightweight model (YOLOv11 Nano)
model = YOLO('yolo11n.pt')

# Train the model on your dataset
results = model.train(
    data="/content/extracted_files/FOSSBot Object Detection/yolo_dataset/data.yaml",
    epochs=25,
    imgsz=640,
    batch=64,
    device=0,
    workers=4,
    cache=True
)

# Verify the results

The code snippet below performs inference using the trained model on five images from the test subset. You can rerun the cell multiple times to see the results on different images.

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# Load the trained model
# If you run the training multiple times, then update the path to train-2, train-3 etc
model_path = '/content/runs/detect/train/weights/best.pt'

if os.path.exists(model_path):
    model = YOLO(model_path)
    print("Loaded the trained model!")
else:
    raise FileNotFoundError(f"Could not find your trained weights at '{model_path}'. Did you finish training?")

# Get a list of all images from the test subset
test_images_dir = '/content/extracted_files/FOSSBot Object Detection/yolo_dataset/images/test'
all_test_files = os.listdir(test_images_dir)
test_images = []

for filename in all_test_files:
    if filename.endswith('.jpg') or filename.endswith('.jpeg') or filename.endswith('.png'):
        # Combine the folder path and filename together
        full_path = os.path.join(test_images_dir, filename)
        test_images.append(full_path)

test_images = [os.path.join(test_images_dir, img) for img in os.listdir(test_images_dir) if img.endswith(('.jpg', '.jpeg', '.png'))]

# Select a few random images to visualize
num_samples = min(5, len(test_images))
random_samples = random.sample(test_images, num_samples)

# Run inference and plot the results
for img_path in random_samples:
    # Run prediction
    results = model.predict(source=img_path, verbose=True)

    # Grab the first result
    result = results[0]

    # result.plot() returns a BGR numpy array with bounding boxes and labels drawn on it
    # We convert BGR to RGB so matplotlib displays the colors correctly
    annotated_img = result.plot()
    annotated_img_rgb = Image.fromarray(annotated_img[..., ::-1])

    # Display the image using matplotlib
    plt.figure(figsize=(10, 7))
    plt.imshow(annotated_img_rgb)
    plt.axis('off')
    plt.title(f"Prediction for: {os.path.basename(img_path)}")
    plt.show()

# Download the trained weights

Navigate to `runs/detect/train/weights` and download the `best.pt` file.